In [16]:
from pymongo import MongoClient
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Conectar a MongoDB
client = MongoClient('mongodb://database:27017/')
db = client['proyecto_bigdata']
coleccion = db['datos_libros']

# Crear SparkSession
spark = SparkSession.builder \
    .appName("G1_Ecommerce_Mystery") \
    .getOrCreate()

print("✓ Spark inicializado")

✓ Spark inicializado


In [17]:
# Obtener datos de MongoDB
datos = list(coleccion.find({"grupo": "G1_Ecommerce"}))
print(f"✓ {len(datos)} registros en MongoDB")

# Limpiar: remover _id que Spark no entiende
datos_limpios = [{k: v for k, v in doc.items() if k != '_id'} for doc in datos]

# Convertir a DataFrame de Spark
df = spark.createDataFrame(datos_limpios)

print(f"✓ DataFrame: {df.count()} registros")

✓ 40 registros en MongoDB
✓ DataFrame: 40 registros


In [18]:
print("\nEsquema de datos:")
df.printSchema()

print("\nPrimeras 5 filas:")
df.show(5, truncate=False)


Esquema de datos:
root
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- valor: double (nullable = true)


Primeras 5 filas:
+---------+-------------------+------------+-----------------------------------------------+-----+
|categoria|fecha_captura      |grupo       |item                                           |valor|
+---------+-------------------+------------+-----------------------------------------------+-----+
|Mystery  |2026-04-14 03:16:51|G1_Ecommerce|Sharp Objects                                  |47.82|
|Mystery  |2026-04-14 03:16:51|G1_Ecommerce|In a Dark, Dark Wood                           |19.63|
|Mystery  |2026-04-14 03:16:51|G1_Ecommerce|The Past Never Ends                            |56.5 |
|Mystery  |2026-04-14 03:16:51|G1_Ecommerce|A Murder in Time                               |16.64|
|Mystery  |2026-04-14 03:16:51|G1_Ecommerce|The Murder of Roger Ack

In [19]:
from pyspark.sql.functions import col

df_filtrado = df.filter(
    (col("item").isNotNull()) & (col("valor") > 0)
)

print("\nLIMPIEZA COMPLETADA")
print(f"   Registros originales: {df.count()}")
print(f"   Registros válidos: {df_filtrado.count()}")


LIMPIEZA COMPLETADA
   Registros originales: 40
   Registros válidos: 40


In [20]:
print("\na) Selección de columnas:")
df.select("item", "valor").show(5)

print("\nb) Productos que cuestan más de £15:")
df.filter(df["valor"] > 15).show(5)

print("\nc) Cantidad de productos por grupo:")
df.groupBy("grupo").count().show()


a) Selección de columnas:
+--------------------+-----+
|                item|valor|
+--------------------+-----+
|       Sharp Objects|47.82|
|In a Dark, Dark Wood|19.63|
| The Past Never Ends| 56.5|
|    A Murder in Time|16.64|
|The Murder of Rog...| 44.1|
+--------------------+-----+
only showing top 5 rows


b) Productos que cuestan más de £15:
+---------+-------------------+------------+--------------------+-----+
|categoria|      fecha_captura|       grupo|                item|valor|
+---------+-------------------+------------+--------------------+-----+
|  Mystery|2026-04-14 03:16:51|G1_Ecommerce|       Sharp Objects|47.82|
|  Mystery|2026-04-14 03:16:51|G1_Ecommerce|In a Dark, Dark Wood|19.63|
|  Mystery|2026-04-14 03:16:51|G1_Ecommerce| The Past Never Ends| 56.5|
|  Mystery|2026-04-14 03:16:51|G1_Ecommerce|    A Murder in Time|16.64|
|  Mystery|2026-04-14 03:16:51|G1_Ecommerce|The Murder of Rog...| 44.1|
+---------+-------------------+------------+--------------------+-----+
o

In [21]:
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("\nANÁLISIS DE MERCADO POR CATEGORÍA:")
reporte_categorias.show()


ANÁLISIS DE MERCADO POR CATEGORÍA:
+---------+---------------+---------------+-------------+-------------+
|categoria|cantidad_libros|precio_promedio|precio_minimo|precio_maximo|
+---------+---------------+---------------+-------------+-------------+
|  Mystery|             40|         32.794|        10.69|        59.48|
+---------+---------------+---------------+-------------+-------------+



In [22]:
ganador = df.orderBy(F.desc("valor")).limit(1)

print("\nEL DETECTIVE DE PRECIOS:")
print("=" * 60)
ganador.select("item", "valor", "categoria", "grupo", "fecha_captura").show(truncate=False)

# Extraer valores para mostrar formateado
resultado = ganador.select("item", "valor", "categoria", "fecha_captura").collect()[0]

print("\nPRODUCTO MÁS CARO:")
print(f"   Nombre: {resultado['item']}")
print(f"   Precio: £{resultado['valor']}")
print(f"   Categoría: {resultado['categoria']}")
print(f"   Fecha captura: {resultado['fecha_captura']}")
print("=" * 60)


EL DETECTIVE DE PRECIOS:
+-----------------------------+-----+---------+------------+-------------------+
|item                         |valor|categoria|grupo       |fecha_captura      |
+-----------------------------+-----+---------+------------+-------------------+
|Boar Island (Anna Pigeon #19)|59.48|Mystery  |G1_Ecommerce|2026-04-14 03:16:52|
+-----------------------------+-----+---------+------------+-------------------+


PRODUCTO MÁS CARO:
   Nombre: Boar Island (Anna Pigeon #19)
   Precio: £59.48
   Categoría: Mystery
   Fecha captura: 2026-04-14 03:16:52


In [23]:
ticket_salida = df.filter(F.col("grupo") == "G1_Ecommerce") \
    .groupBy("categoria") \
    .agg(
        F.count("item").alias("total_libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"\n--- TICKET DE SALIDA: G1_Ecommerce ---")
ticket_salida.show()


--- TICKET DE SALIDA: G1_Ecommerce ---
+---------+------------+------------+---------------------+
|categoria|total_libros|precio_medio|ultima_sincronizacion|
+---------+------------+------------+---------------------+
|  Mystery|          40|       32.79|  2026-04-14 03:16:57|
+---------+------------+------------+---------------------+

